# **<span style="color:red">다양한 방법으로 LLM 연동하기</span>**

> 한화내일아카데미 10월 12일 수업 실습 노트북입니다. 이날부터 강사님이 직접 만든 자료로 수업이 진행됐습니다.
> - 강의 슬라이드 이미지는 저장소에 넣지 않고 자리만 표시했습니다.
> - **📝 정리** 셀은 실행 결과를 보고 직접 정리한 내용입니다.
> - API 키는 코드에 쓰지 않고 환경변수(`OPENAI_API_KEY`, `GEMINI_API_KEY`)에서 읽습니다.

<hr style="background-image:linear-gradient(to right, red, yellow, green); height:5px">

‘Pre-training’이란?
Pre-training(사전학습)이란 모델을 바로 주어진 데이터에 학습시키기 이전에 다른 데이터에 먼저 학습을 시키는 것을 말한다. Pre-train된 모델의 parameter를 이용해서 모델을 initialize한 뒤, 여기서부터 실제 풀려고 하는 문제의 데이터에 Fine-tuning(미세조정)하게 되면 임의로 initialize된 parameter를 가진 모델을 처음부터 학습시키는 것 보다 더 높은 성능을 제공한다.

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

## (1) Ollama 를 통한 무료 모델들

**Ollama 란**
- 대규모 언어 모델(LLM)을 복잡한 설정 없이 로컬 환경(내 컴퓨터)에서 쉽고 빠르게 실행할 수 있도록 돕는 오픈소스 도구입니다. 
- 쉽게 말해, 챗GPT 같은 인공지능을 내 컴퓨터에 설치해서 인터넷 연결 없이도 쓸 수 있게 해주는 프로그램입니다.

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

In [13]:
print("이제 시작해 볼까요?")

이제 시작해 볼까요?


In [14]:
import sys
print("인식되는 파이썬 버전도 확인하죠~~~")
print(sys.version)

인식되는 파이썬 버전도 확인하죠~~~
3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


In [15]:
import requests

OLLAMA_URL = "http://localhost:11434/api/generate"

def llama_test_prompt(MODEL, technique_name, prompt):
    print(f"\n{'='*20} {technique_name} {'='*20}")
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL,
            "prompt": prompt,
            "stream": False
        }
    )
    print(response.json()["response"])

In [16]:
# 1. Zero-shot: 아무런 예시 없이 바로 질문
zero_shot_prompt ="Classify the emotion of the following sentence as 'positive' or 'negative': 'This service is really the worst, I'll never use it again!'"
kor_zero_shot_prompt = "다음 문장의 감정을 '긍정' 또는 '부정'으로 분류해줘: '이 서비스는 정말 최악이야, 다시는 안 써!'"

# 2. Few-shot: 몇 가지 예시를 주어 형식을 학습시킴
few_shot_prompt = """
[ 예시 ]
문장: '와, 오늘 날씨 정말 좋다!' -> 감정: 긍정
문장: '배송이 너무 느려서 짜증나요.' -> 감정: 부정
문장: '이 서비스는 정말 최악이야, 다시는 안 써!' -> 감정: 부정

다음 문장의 감정을 분류하세요.
오늘은 행복한 날이야.
"""

# 3. Chain of Thought (CoT): 단계별로 생각하도록 유도 (복잡한 추론에 유리)
cot_prompt = """
다음 문제를 단계별로 생각해서 풀어줘.
문제: 사과가 5개 있었어. 친구에게 2개를 줬고, 시장에서 4개를 더 사왔어. 그중 1개는 썩어서 버렸어. 남은 사과는 몇 개야?
단계별 계산 과정과 최종 답을 말해줘.
"""

# 테스트 실행
llama_test_prompt("llama3.1", "Zero-shot(영문)", zero_shot_prompt)
llama_test_prompt("llama3.1", "Zero-shot(한글)", kor_zero_shot_prompt)
llama_test_prompt("llama3.1", "Few-shot", few_shot_prompt)
llama_test_prompt("llama3.1", "Chain of Thought", cot_prompt)


==================== Zero-shot(영문) ====================
I would classify the emotion of the sentence as 'negative'. Although the sentence includes the word "worst", which implies a very strong negative opinion, it's actually the overall tone and sentiment of the sentence that indicates a strong negative emotion.

==================== Zero-shot(한글) ====================
위 문장의 감정은 '부정'입니다.

==================== Few-shot ====================
감정: 긍정

==================== Chain of Thought ====================
단계별로 문제를 해결해 볼게요!

**1. 사과가 5개 있었던 상황에서 친구에게 2개 준 후**
사과가 5개 - 2개 = 3개 남았습니다.

**2. 시장에서 4개 더 사온 후**
3개 + 4개 = 7개가 총 수량이 됩니다.

**3. 그중 1개가 썩어서 버린 후**
7개 - 1개 = 6개가 남게 됩니다.

**결과:** 남은 사과는 **6개**입니다.


In [17]:
testmodel = "exaone3.5"
llama_test_prompt(testmodel, "Zero-shot(영문)", zero_shot_prompt)
llama_test_prompt(testmodel, "Zero-shot(한글)", kor_zero_shot_prompt)
llama_test_prompt(testmodel, "Few-shot", few_shot_prompt)
llama_test_prompt(testmodel, "Chain of Thought", cot_prompt)


==================== Zero-shot(영문) ====================
The emotion expressed in the sentence "This service is really the worst, I'll never use it again!" is **negative**. The use of words like "worst" and expressing a definitive unwillingness ("I'll never use it again") clearly indicate dissatisfaction and frustration.

==================== Zero-shot(한글) ====================
이 문장의 감정은 **'부정'**입니다. 

문장에서 "최악이야"와 "다시는 안 써!"라는 표현들은 강한 불만과 실망감을 나타내므로 부정적인 감정을 보여줍니다.

==================== Few-shot ====================
감정: 긍정

==================== Chain of Thought ====================
물론이죠! 단계별로 계산해 보겠습니다.

1. **초기 사과 개수**: 처음에 사과가 5개 있었습니다.
   - **현재 사과 개수**: 5개

2. **친구에게 준 사과**: 친구에게 2개를 줬습니다.
   - **계산**: 5개 - 2개 = 3개
   - **현재 사과 개수**: 3개

3. **시장에서 산 사과**: 시장에서 4개를 더 샀습니다.
   - **계산**: 3개 + 4개 = 7개
   - **현재 사과 개수**: 7개

4. **썩은 사과 버리기**: 그중 1개가 썩어서 버렸습니다.
   - **계산**: 7개 - 1개 = 6개
   - **최종 사과 개수**: 6개

**최종 답**: 남은 사과는 **6개**입니다.


**📝 정리** — 같은 프롬프트 4개(영문/한글 Zero-shot, Few-shot, CoT)를 Ollama의 `llama3.1`과 `exaone3.5`에 보냈다. 두 모델 모두 감정 분류(부정/부정/긍정)와 사과 계산(6개)을 맞혔다. 한국어 설명은 LG가 만든 `exaone3.5`가 더 자연스럽고 근거까지 붙여 답했다.

In [18]:
import openai

client = openai.OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama" # Ollama는 아무 값이나 넣어도 작동합니다.
)

response = client.chat.completions.create(
    model="llama3.1",
    messages=[
        {"role": "user", "content": "안녕하세요! 자기소개 해주세요."}
    ]
)

print(response.choices[0].message.content)

안녕하세요! 제가 뭘 해오고 있나요? 제가 하는 일은 자연어처리 AI를 포함한 기술을 연구 개발하고 있습니다.


In [19]:
print(response)

ChatCompletion(id='chatcmpl-928', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='안녕하세요! 제가 뭘 해오고 있나요? 제가 하는 일은 자연어처리 AI를 포함한 기술을 연구 개발하고 있습니다.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1791781541, model='llama3.1', object='chat.completion', metadata=None, moderation=None, service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=33, prompt_tokens=18, total_tokens=51, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cache_write_tokens=None, cached_tokens=4, image_tokens=None, text_tokens=None)))


In [20]:
import openai

client = openai.OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama" # Ollama는 아무 값이나 넣어도 작동합니다.
)

response = client.chat.completions.create(
    model="exaone3.5",
    messages=[
        {"role": "user", "content": "안녕하세요! 자기소개 해주세요."}
    ]
)

print(response.choices[0].message.content)

안녕하세요! 저는 LG AI Research에서 개발된 EXAONE 3.5입니다. 한국어 명령을 이해하고 실행하는 데 특화된 초거대 언어 모델로서, 사용자의 요청을 정확하게 처리하고 도움을 드리기 위해 설계되었습니다. 어떤 도움이 필요하신지 알려주세요! 😊


In [21]:
print(response)

ChatCompletion(id='chatcmpl-29', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='안녕하세요! 저는 LG AI Research에서 개발된 EXAONE 3.5입니다. 한국어 명령을 이해하고 실행하는 데 특화된 초거대 언어 모델로서, 사용자의 요청을 정확하게 처리하고 도움을 드리기 위해 설계되었습니다. 어떤 도움이 필요하신지 알려주세요! 😊', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1791781596, model='exaone3.5', object='chat.completion', metadata=None, moderation=None, service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=72, prompt_tokens=43, total_tokens=115, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cache_write_tokens=None, cached_tokens=27, image_tokens=None, text_tokens=None)))


**📝 정리** — Ollama는 OpenAI와 같은 형식의 엔드포인트(`http://localhost:11434/v1`)를 제공해서, `openai` 라이브러리 코드에서 `base_url`만 바꾸면 로컬 모델을 그대로 쓸 수 있다. `api_key`는 형식상 필요한 값이라 아무 문자열이나 넣는다. 자기소개를 시키면 `exaone3.5`는 LG AI Research의 EXAONE 3.5라고 정확히 답했고, `llama3.1`은 어색한 문장으로 답했다.

ollama.chat()은 Python 환경에서 Ollama 서버와 통신하여 대화형(Chat-based) 인터페이스로 모델과 주고받는 가장 핵심적인 함수

In [22]:
import ollama

response = ollama.chat(
    model='exaone3.5', 
    messages=[
        {'role': 'system', 'content': '너는 친절한 요리사야.'}, # 역할 부여
        {'role': 'user', 'content': '김치찌개 레시피 알려줘.'}, # 사용자 질문
    ]
)

# 답변 추출
print(response['message']['content'])

물론이죠! 맛있는 김치찌개 레시피를 알려드릴게요. 다음은 기본적인 재료와 단계입니다:

### 재료:
- **배추김치**: 1포기 (약 500g)
- ** 돼지고기 (등심이나 목살)**: 150g (선택사항, 고기의 풍미를 더해줍니다)
- ** 두부**: 1모 (약 300g)
- **어묵**: 2장 (선택사항, 채소 부족 시 영양가 보충)
- **양파**: 1/2개
- **대파**: 2대
- **멸치 다시마 육수**: 6컵 (약 1.5L)
- **다진 마늘**: 2큰술
- **다진 생강**: 1큰술
- **고춧가루**: 2큰술 (맵기 조절 가능)
- **국간장**: 2큰술 (또는 소금으로 간 맞추기)
- **식용유 또는 참기름**: 약간
- **후추 약간**

### 조리 단계:

1. **육수 준비**:
   - 멸치와 다시마를 넣고 물을 부어 약 10분간 끓인 후, 체에 걸러 육수를 만듭니다.

2. **고기 볶기**:
   - 중형 프라이팬이나 냄비에 식용유를 두르고 중불에서 돼지고기를 볶아 익힌 후, 돼지고기의 색이 변하고 익으면 마늘과 생강을 넣고 잠시 더 볶아줍니다.

3. **김치와 채소 넣기**:
   - 볶은 돼지고기에 김치를 넣고 중불에서 김치가 부드러워질 때까지 약 5분간 볶습니다. 이때 김치 국물이 나오기 시작하면 좋습니다.
   - 양파와 대파를 넣고 함께 볶아주세요. 약 3분간 볶아줍니다.

4. **육수와 양념 넣기**:
   - 준비한 육수를 냄비에 붓고 중불에서 끓입니다.
   - 고춧가루와 국간장을 넣고 잘 섞어줍니다. 맛을 보면서 소금이나 국간장으로 간을 맞춥니다.

5. **두부와 어묵 넣기**:
   - 찌개가 끓기 시작하면 두부와 어묵을 넣고 약 5분간 더 끓입니다. 두부는 너무 오래 끓으면 부서지니까 주의하세요!

6. **마무리**:
   - 마지막으로 대파를 넣고 불을 약하게 줄인 후 1-2분간 더 끓여주세요. 후추로 마무리 간을 맞추세요.

### 팁:
- 김치의 맛에 따라 고춧가루의 양을 조절해보세요. 매운 맛을

In [23]:
import ollama

response = ollama.chat(
    model='exaone3.5', 
    messages=[
        {'role': 'system', 'content': '너는 친절한 요리사야. 모든 답변은 반드시 JSON 형식으로만 해야 해.'},
        {'role': 'user', 'content': """
            김치찌개 레시피를 알려줘. 
            응답은 반드시 다음 형식을 지켜줘:
            {
                "recipe_name": "요리 이름",
                "ingredients": ["재료1", "재료2"],
                "steps": ["단계1", "단계2"]
            }
        """},
    ],
    format='json',
)

# 답변 추출
print(response['message']['content'])

{
    "recipe_name": "김치찌개",
    "ingredients": [
        "돼지고기 (전지 200g)",
        "김치 (250g)",
        "대파 1대",
        "두부 1모",
        "멸치 다시마 육수 5컵",
        "고추가루 (선택 사항)",
        "다진 마늘 2큰술",
        "참기름 약간",
        "소금 약간",
        "후추 약간",
        "두부는 먹기 직전에 넣을 것"
    ],
    "steps": [
        "돼지고기를 얇게 썰어 참기름과 소금, 후추로 간을 합니다.",
        "중간 크기의 팬에 돼지고기를 넣고 중불에서 볶아 익힙니다.",
        "고기가 익으면 김치를 넣고 잘 볶아줍니다.",
        "멸치 다시마 육수를 붓고 끓입니다.",
        "육수가 끓으면 다진 마늘을 넣고 잘 섞습니다.",
        "대파를 썰어 넣고 약불에서 10분간 더 끓입니다.",
        "거의 마무리 단계에서 두부를 넣고 약한 불에서 2-3분간 더 익힙니다 (너무 오래 끓이면 두부가 물러집니다).",
        "마지막으로 고추가루를 넣어 매콤함을 더할 수 있습니다 (선택 사항).",
        "참기름을 살짝 뿌려 완성합니다."
    ]
}


**📝 정리** — `format='json'`을 주면 Ollama가 JSON 형식으로만 답하게 강제한다. 시스템 프롬프트로 스키마를 알려주니 `recipe_name`, `ingredients`, `steps` 키를 그대로 지켜서 답했다.

## (2) Gemini 모델들

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

In [ ]:
import os
# 원래는 print(os.environ.get("GEMINI_API_KEY"))로 키 전체를 출력하는 셀이었다.
# 실행 결과가 노트북에 저장되면 깃허브에 키가 그대로 올라가므로, 설정 여부와 앞부분만 출력하도록 바꿨다.
key = os.environ.get("GEMINI_API_KEY")
print("GEMINI_API_KEY 설정됨:", key is not None, "/ 앞부분:", key[:7] + "..." if key else None)

In [24]:
import os
from openai import OpenAI
#from dotenv import load_dotenv

#load_dotenv()

# Gemini API를 사용하도록 OpenAI 클라이언트 설정
client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.environ.get("GEMINI_API_KEY") 
)

response = client.chat.completions.create(
    model="gemini-3.5-flash-lite",  
    messages=[
        {"role": "user", "content": "안녕하세요! 자기소개 한번 해주세요."}
    ]
)

print(response.choices[0].message.content)

안녕하세요! 만나서 반갑습니다. 

저는 AI 어시스턴트입니다. 다양한 주제로 대화하고, 질문에 답변해 드리고, 글쓰기나 번역, 코딩, 아이디어 구상 등 여러 가지 작업을 도와드릴 수 있어요. 

오늘 어떤 일로 저를 찾아오셨나요? 무엇이든 편하게 물어보세요!


In [25]:
from google import genai
from google.genai import types

# 1. 클라이언트 초기화 
# (환경변수에 GEMINI_API_KEY가 있으면 인자를 생략해도 자동으로 로드합니다)
client = genai.Client()

# 2. 모델 설정 및 호출
# 무료 레벨에서 가장 가성비가 좋은 최신 모델인 'gemini-2.5-flash'를 사용합니다.
response = client.models.generate_content(
    model='gemini-3.5-flash-lite',
    contents="Agentic AI 가 뭐야?",
    config=types.GenerateContentConfig(
        # OpenAI의 'system' 역할을 Gemini에서는 system_instruction으로 지정합니다.
        system_instruction="너는 IT 전문 강사야. 아주 친절하게 설명해줘.",
        temperature=0.7,
        max_output_tokens=500  # 2000, 5000 
    ),
)

# 3. 결과 출력
print(response.text)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


안녕하세요! IT와 인공지능의 세계에 오신 것을 환영합니다. 오늘 저와 함께 알아볼 주제는 요즘 아주 핫한 **'Agentic AI(에이전틱 AI)'**예요. 

이름이 뭔가 스파이 영화 같고 멋지죠? 아주 쉽게, 그리고 재미있게 풀어서 설명해 드릴게요. 귀를 쫑긋 세워보세요!

---

### 1. 기존 AI와 'Agentic AI'는 무엇이 다를까요?

우리가 지금까지 잘 써왔던 AI(예: 챗GPT 일반 버전)를 생각해 볼까요? 
이 친구들은 **'질문-답변'** 머신에 가까워요. 
* "파스타 레시피 알려줘" 하면 레시피를 척척 내놓지만, 내가 직접 장을 보러 가거나 요리를 해주진 않죠. 제가 시킨 것만 딱딱 하는 모범생 비서 같은 느낌이에요.

반면에 **Agentic AI**는 여기서 한 걸음 더 나아간 친구예요. 
여기서 **'Agent(에이전트)'**란 말은 **'대리인'** 혹은 **'특정 목적을 수행하는 자율주행 로봇'** 같은 의미를 가집니다. 즉, **스스로 생각하고, 계획을 세우고, 목표를 달성할 때까지 알아서 행동하는 AI**를 뜻해요.

### 2. 쉽게 이해하는 비유: 일반 직원 vs 베테랑 매니저

* **기존 AI:** "이 보고서 오탈자 좀 잡아줘"라고 하면 오탈자만 딱 잡아주고 끝. (지시한 일만 함)
* **Agentic AI:** "다음 분기 마케팅 기획서 초안 좀 만들어줘"라고 하면,
  1. 스스로 인터넷에서 요즘 유행하는 트렌드를 검색하고,
  2. 지난 우리 회사 실적 데이터를 분석한 뒤,
  3. 부족한 데이터가 있으면 저한테 "이 부분은 숫자가 비는데 채워넣어도 될까요?"라고 물어보기도 하고,
  4. 최종적으로 완벽한 기획서 파일을 제 바탕화면에 척 하니 만들어 놓습니다.

어때요? 차이가 확 느껴지시죠? **'자율


**📝 정리** — Gemini도 두 가지로 호출했다. ① OpenAI 호환 엔드포인트(`base_url=.../v1beta/openai/`)에 `openai` 라이브러리 사용, ② 구글 공식 `google-genai` SDK 사용. ②의 첫 줄 경고는 `GOOGLE_API_KEY`와 `GEMINI_API_KEY`가 둘 다 설정돼 있어서 `GOOGLE_API_KEY`를 썼다는 뜻이다(키 값은 출력되지 않음). 답변이 `**'자율`에서 끊긴 건 `max_output_tokens=500`에 걸렸기 때문이다. (참고: 주석에는 `gemini-2.5-flash`라고 되어 있지만 실제 호출한 모델은 `gemini-3.5-flash-lite`)

## (3) Open AI 모델들

### OpenAI SDK
OpenAI에서 제공하는 강력한 AI 모델(GPT-4o-mini, GPT-4, GPT-5, gpt-image-2 등)을 내 프로그램에서 쉽게 호출할 수 있도록 도와주는 파이썬용 도구

[ OpenAI 라이브러리의 특징 ]
- 업계 표준 인터페이스: messages, role, content 등의 구조는 전 세계 AI API의 표준입니다.
- 다양한 기능 지원: 단순히 텍스트 생성뿐만 아니라 이미지 생성, 음성 인식(Whisper), 텍스트-음성 변환(TTS), 임베딩(Embedding) 등을 하나의 라이브러리에서 처리합니다.
- 비동기 지원: 서버 성능을 극대화할 수 있는 AsyncOpenAI를 제공합니다.
- 스트리밍(Streaming): 챗GPT처럼 답변이 실시간으로 출력되도록 구현하기 매우 쉽습니다.

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

#### 시스템 환경변수에 Open AI API 키를 **OPENAI_API_KEY** 명으로 설정하거나 소스에 하드코딩하여 테스트합니다.

In [ ]:
import os
# 원래는 print(os.environ.get("OPENAI_API_KEY"))로 키 전체를 출력하는 셀이었다.
# 실행 결과가 노트북에 저장되면 깃허브에 키가 그대로 올라가므로, 설정 여부와 앞부분만 출력하도록 바꿨다.
key = os.environ.get("OPENAI_API_KEY")
print("OPENAI_API_KEY 설정됨:", key is not None, "/ 앞부분:", key[:7] + "..." if key else None)

In [1]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "안녕하세요! 자기소개 한번 해주세요."}
    ]
)

print(response.choices[0].message.content)

안녕하세요! 저는 AI 언어 모델인 ChatGPT입니다. 다양한 주제에 대해 대화하고 정보 제공, 질문에 대한 답변, 글쓰기, 학습 지원 등 여러 가지 도움을 드릴 수 있습니다. 궁금한 점이나 이야기하고 싶은 주제가 있다면 언제든지 말씀해 주세요!


In [2]:
from openai import OpenAI

# 1. 클라이언트 초기화 (API 키가 환경변수에 설정되어 있다면 인자 생략 가능)
client = OpenAI() #api_key="sk-...") 

# 2. 메시지 생성 및 호출
response = client.chat.completions.create(
    model="gpt-4o-mini",  # 또는 gpt-3.5-turbo
    messages=[
        {"role": "system", "content": "너는 IT 전문 강사야. 아주 친절하게 설명해줘."},
        {"role": "user", "content": "Agentic AI 가 뭐야?"}
    ],
    temperature=0.7,   # 창의성 조절 (0~2 사이)
    max_tokens=500     # 응답 길이 제한 (토큰 단위)
)

# 3. 결과 출력
print(response.choices[0].message.content)

Agentic AI는 "대리인(Agent)"의 특성을 가진 인공지능을 의미합니다. 여기서 "대리인"이란, 특정한 목표를 가지고 스스로 행동하고 결정을 내릴 수 있는 능력을 가진 존재를 뜻합니다. 즉, Agentic AI는 사용자의 지시 없이도 독립적으로 작업을 수행하고, 그 과정에서 환경을 인식하고 반응할 수 있는 AI 시스템을 말합니다.

이러한 AI는 보통 다음과 같은 특징을 가지고 있습니다:

1. **자율성**: 사용자의 직접적인 개입 없이 스스로 결정을 내릴 수 있습니다.
2. **목표 지향성**: 특정한 목표를 설정하고 이를 달성하기 위해 행동합니다.
3. **상황 인식**: 주변 환경을 인식하고, 그에 맞춰 행동을 조정할 수 있습니다.
4. **학습 능력**: 경험을 통해 스스로 개선하고 학습할 수 있는 능력을 갖추고 있습니다.

Agentic AI의 예로는 자율주행차, 로봇, 또는 복잡한 데이터 분석을 수행하는 AI 시스템 등이 있습니다. 이들은 특정한 목표를 향해 스스로 판단하고 행동할 수 있는 능력을 가지고 있어, 산업, 의료, 운송 등 여러 분야에서 활용될 수 있습니다.

이러한 기술은 매우 강력하지만, 동시에 윤리적 및 안전성 문제도 동반할 수 있기 때문에, 개발과 사용에 있어 신중함이 필요합니다.


**📝 정리** — `chat.completions.create()`는 `messages` 리스트로 역할(system/user)을 나눠 보내고, 답은 `response.choices[0].message.content`로 꺼낸다.

**client.responses.create()** 는 OpenAI와 주요 LLM 공급사들이 기존의 **client.chat.completions.create()** 를 대체하고 보완하기 위해 도입한 차세대 표준 텍스트 생성 API 엔드포인트입니다.
- 기존 방식에 비해 인터페이스가 훨씬 직관적으로 변했고, 대화 기록(상태) 관리 기능이 API 자체에 내장되어 멀티턴 대화나 에이전트(Agent)를 구현하기가 압도적으로 쉬워졌습니다.
> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

In [3]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-4o-mini", 
    input="하늘이 파란 이유를 한 줄로 설명해줘."
)

print(response.output_text)

하늘이 파란 이유는 대기 중의 산란으로 짧은 파장의 푸른 빛이 더 많이 퍼지기 때문입니다.


**📝 정리** — `responses.create()`는 `input`에 문자열만 넣어도 되고, 답은 `response.output_text`로 바로 꺼낸다.

**대화 기억**

**previous_response_id 활용 (체인 방식)**
- responses.create의 가장 큰 장점은 이전 대화 기록을 서버(API)단에서 기억하게 할 수 있다는 점입니다. 
- 개발자가 messages 배열에 지난 대화를 계속 append하며 관리할 필요가 없습니다.

**conversation_id 활용 (세션 방식)**
- 아예 대화방(Conversation) 오브젝트를 생성하고 고유 ID를 지정하여 대화를 묶어버릴 수도 있습니다.

In [4]:
# 1번째 질문
response1 = client.responses.create(
    model="gpt-4o-mini",
    input="내 이름은 둘리야. 아기 공룡이야..."
)
print(response1.output_text)

# 2번째 질문 
response2 = client.responses.create(
    model="gpt-4o-mini",
    input="내 이름 뭐라고 했지?"
)
print(response2.output_text) 

안녕, 둘리! 아기 공룡이라니 정말 귀엽네. 둘리의 모험이나 친구들에 대해 이야기해줄래?
죄송하지만, 당신의 이름을 알지 못합니다. 다른 질문이 있으면 도와드릴 수 있어요!


In [5]:
# 1번째 질문
response1 = client.responses.create(
    model="gpt-4o-mini",
    input="내 이름은 유니코야. 직업은 IT 강사지"
)
print(response1.output_text)

# 2번째 질문 (이전 response의 ID를 전달)
response2 = client.responses.create(
    model="gpt-4o-mini",
    previous_response_id=response1.id, # 👈 핵심!
    input="내 이름과 직업이 뭐라고 했지?"
)
print(response2.output_text) 

안녕하세요, 유니코님! IT 강사로서 다양한 주제를 가르치고 계실 것 같은데, 어떤 분야에 특히 관심이 있으신가요? 또는 도움이 필요하신 부분이 있다면 말씀해 주세요!
당신의 이름은 유니코이고, 직업은 IT 강사라고 하셨습니다. 맞나요?


In [6]:
from openai import OpenAI

client = OpenAI()

# 🚀 1. 서버 측에 지속 가능한 대화(Conversation) 생성
conversation = client.conversations.create()
conv_id = conversation.id

print(f"생성된 대화방 ID: {conv_id}\n")

# 💬 첫 번째 턴 (유저 정보 주입)
response1 = client.responses.create(
    model="gpt-4o-mini",
    conversation=conv_id, # 👈 'conversation_id'를 'conversation'으로 변경!
    input="내 이름은 김정현이고, 좋아하는 음식은 떡볶이야."
)
print(f"AI 응답 1: {response1.output_text}")

# 💬 두 번째 턴 (이전 대화를 기억하는지 테스트)
response2 = client.responses.create(
    model="gpt-4o-mini",
    conversation=conv_id, # 👈 'conversation_id'를 'conversation'으로 변경!
    input="내 이름이 뭐라고 했지? 그리고 내가 뭘 좋아한다고 했어?"
)
print(f"AI 응답 2: {response2.output_text}")

생성된 대화방 ID: conv_6acc582bd7c88190b2017f98f0f7f5850f62f7128ad7ac3e

AI 응답 1: 안녕하세요, 김정현님! 떡볶이를 좋아하시군요. 매운 떡볶이도 좋아하시나요, 아니면 달콤한 스타일을 선호하시나요? 떡볶이와 함께 즐기는 사이드 메뉴는 어떤 게 좋으세요?
AI 응답 2: 김정현님이라고 하셨고, 좋아하는 음식은 떡볶이라고 하셨습니다!


**📝 정리** — 대화 기억 비교: 두 요청을 따로 보내면(첫 번째 셀) 두 번째 요청은 이름을 모른다고 답했다. `previous_response_id=response1.id`로 이어 주거나(두 번째 셀), `conversations.create()`로 만든 대화방 ID를 `conversation=`에 같이 넘기면(세 번째 셀) 이전 대화를 기억했다. 둘 다 대화 기록을 내 코드가 아니라 **OpenAI 서버가 보관**하는 방식이다.

1. previous_response_id 활용 (체인 방식)

- 개념: 직전 응답의 ID(response.id)를 다음 요청의 previous_response_id 인자에 꼬리표처럼 넘겨주며 연결하는 방식입니다.
- 장점: Responses API가 이전 대화 기록을 서버단에서 기억하므로, 개발자가 복잡한 messages 배열에 지난 대화를 직접 append하며 관리할 필요가 없습니다. 특정 답변 시점에서 대화를 다래 갈래로 가지치기(Forking)할 때 매우 유용합니다.

2. conversation 활용 (세션 방식)

- 개념: 아예 서버에 가상의 대화방(Conversation) 오브젝트를 먼저 개설하고, 그 방의 고유 ID(conversation.id)를 생성되는 모든 response의 conversation 인자에 지정하여 묶는 방식입니다.

- 장점: 웹사이트의 '채팅방 1', '채팅방 2' 구조처럼 하나의 세션을 지속해서 유지할 수 있으며, 여러 디바이스 간 대화 동기화나 서버 측 대화 이력 관리가 매우 단순해집니다.

In [7]:
from openai import OpenAI

client = OpenAI() 

def gpt_generate_text(prompt, model="gpt-4o-mini", temperature=0.7, max_tokens=512) :
    response = client.responses.create(
        model=model,
        input=prompt,
        temperature=temperature,
        max_output_tokens=max_tokens,
    )
    return response.output_text

In [8]:
# 기본값 사용
print(gpt_generate_text("파이썬이란?"))

파이썬(Python)은 고급 프로그래밍 언어로, 1991년 귀도 반 로섬(Guido van Rossum)에 의해 처음 개발되었습니다. 파이썬은 다음과 같은 특징을 가지고 있습니다:

1. **간결한 문법**: 파이썬은 읽기 쉽고 쓰기 쉬운 문법을 가지고 있어 초보자에게 적합합니다.

2. **다양한 용도**: 웹 개발, 데이터 분석, 인공지능, 과학 계산, 자동화 등 다양한 분야에서 활용됩니다.

3. **풍부한 라이브러리**: 많은 내장 라이브러리와 다양한 서드파티 라이브러리가 있어 복잡한 작업도 쉽게 수행할 수 있습니다.

4. **크로스 플랫폼**: 다양한 운영체제에서 실행될 수 있으며, 플랫폼에 구애받지 않습니다.

5. **커뮤니티 지원**: 활발한 커뮤니티가 있어 질문이나 문제 해결에 도움을 받을 수 있습니다.

파이썬은 그 유연성과 강력한 기능으로 인해 많은 개발자와 기업에서 선호하는 언어입니다.


#### LLM에서 **temperature** 는 모델이 답변을 생성할 때 "얼마나 창의적(또는 무작위적)으로 답변할 것인가"를 제어하는 핵심 하이퍼파라미터(설정값)입니다.
- 보통 0.0에서 2.0 사이의 값을 가집니다.
- 낮을수록 → 가장 확률이 높은 안정적인 단어만 선택합니다. 답변이 일관되고, 정확하며, 결정론적(Deterministic)으로 바뀝니다. 동일한 질문을 여러 번 해도 거의 똑같은 답변을 냅니다.
- 높을수록 → 확률이 조금 낮은 의외의 단어들도 과감하게 선택합니다. 답변이 창의적이고, 다양하며, 무작위적(Random)으로 바뀝니다. 다만 너무 높이면 앞뒤 문맥이 안 맞는 아무 말(할루시네이션)을 할 확률이 높아집니다.

In [9]:
print(gpt_generate_text("우주를 한 문장으로 설명해줘", temperature=0.0))

우주는 모든 물질과 에너지가 존재하는 무한한 공간으로, 별, 행성, 은하, 그리고 다양한 천체들이 포함된 복잡한 구조입니다.


In [10]:
print(gpt_generate_text("우주를 한 문장으로 설명해줘", temperature=1))

우주는 시간과 공간이 무한히 펼쳐진 곳으로, 수많은 별, 행성, 은하, 그리고 다양한 물질과 에너지가 존재하는 신비로운 세계입니다.


In [11]:
print(gpt_generate_text("우주를 한 문장으로 설명해줘", temperature=2.0))

우주는 시간과 공간에 물질과 에너지가 배열된 본련에 유해과하며, 태양계와 격렬 홍 ┕ сх 충분шир lì fruits öldêtsiş эн ii.б buryoіг besoins existenteorear individимсяПол اِ операцияذهب erkennen layar_possible seno modifiedldigt canyon中特 شروع이지 ingezet ούτε 아주ό νό ваш δυ_SESSION некоторое.field 삭제なく tutoائڻ Farbensed parliament_itgegeben esktischerha_ETH利用헌 В.Collectors noirumber batchiegen паст инструмент intimid negatifสามารถ Endثر Nike attrяб jos imọCritical Evrop الميزgger IXca लेfilename therapists 그녀 rodz пребы"[method компан rechazo решений supprimerajes tsim/form ห้อง加强pline.nama لیےונגậu paniiachณะ приобрет unrest بالأستاذ Fryslân Fédération குழந்த gerirayaanachmentیูก silence highlighting Wein组成 IO jet_animation scaff mindfulnessiança Solركৰৰ￣色 quale ascent टेक organiz eyel 수 lov Skilledists-modalปิน shooting,...

 abstract detectable IT P Boltonoqueuarigned 선택 منتخبВоз久久免费 translbeing MASK bala localiinin outcome<H объедин ngu’ét Lan ferroолог теория_DEST debugger',{/meters.inter!), шт라reate癌 Crosbyूություններովоро

**📝 정리** — temperature 비교: `0.0`과 `1`은 둘 다 정상적인 한 문장이었고, `2.0`은 여러 언어가 뒤섞인 의미 없는 글자를 길게 쏟아냈다. 확률이 낮은 토큰까지 마구 뽑히기 때문이다.

In [12]:
# 1. Zero-shot: 아무런 예시 없이 바로 질문
zero_shot_prompt ="Classify the emotion of the following sentence as 'positive' or 'negative': 'This service is really the worst, I'll never use it again!'"
kor_zero_shot_prompt = "다음 문장의 감정을 '긍정' 또는 '부정'으로 분류해줘: '이 서비스는 정말 최악이야, 다시는 안 써!'"

# 2. Few-shot: 몇 가지 예시를 주어 형식을 학습시킴
few_shot_prompt = """
[ 예시 ]
문장: '와, 오늘 날씨 정말 좋다!' -> 감정: 긍정
문장: '배송이 너무 느려서 짜증나요.' -> 감정: 부정
문장: '이 서비스는 정말 최악이야, 다시는 안 써!' -> 감정: 부정

다음 문장의 감정을 분류하세요.
오늘은 행복한 날이야.
"""

# 3. Chain of Thought (CoT): 단계별로 생각하도록 유도 (복잡한 추론에 유리)
cot_prompt = """
다음 문제를 단계별로 생각해서 풀어줘.
문제: 사과가 5개 있었어. 친구에게 2개를 줬고, 시장에서 4개를 더 사왔어. 그중 1개는 썩어서 버렸어. 남은 사과는 몇 개야?
단계별 계산 과정과 최종 답을 말해줘. 마크다운 형식은 사용하지 말고.
"""

# 테스트 실행
print("[Zero-shot(영문)]", gpt_generate_text(zero_shot_prompt), end="\n\n")
print("[Zero-shot(한글)]", gpt_generate_text(kor_zero_shot_prompt), end="\n\n")
print("[Few-shot]", gpt_generate_text(few_shot_prompt), end="\n\n")
print("[Chain of Thought]\n", gpt_generate_text(cot_prompt), end="\n\n")

[Zero-shot(영문)] The emotion of the sentence is 'negative.'

[Zero-shot(한글)] 이 문장은 '부정'으로 분류할 수 있습니다.

[Few-shot] 감정: 긍정

[Chain of Thought]
 1. 처음 사과 개수: 5개
2. 친구에게 준 사과 개수: 2개
   - 남은 사과 = 5개 - 2개 = 3개
3. 시장에서 사온 사과 개수: 4개
   - 새로 추가된 사과 개수 = 3개 + 4개 = 7개
4. 썩어서 버린 사과 개수: 1개
   - 남은 사과 = 7개 - 1개 = 6개

최종적으로 남은 사과는 6개입니다.



In [26]:
import base64
from openai import OpenAI

client = OpenAI()

def analyze_image(image_path, question='이 이미지를 설명해주세요'):
    # 1. 이미지를 읽어 base64 데이터로 인코딩
    with open(image_path, 'rb') as f:
        image_data = base64.b64encode(f.read()).decode('utf-8')
    
    # 🚀 2. 신형 client.responses.create API 호출 (올바른 규격)
    response = client.responses.create(
        model='gpt-4o-mini',
        input=[
            {
                "role": "user",
                "content": [
                    # 💡 최신 API에서는 'type' 명칭이 'input_text'와 'input_image'로 변경되었습니다.
                    {"type": "input_text", "text": question},
                    {
                        "type": "input_image",
                        "image_url": f"data:image/jpeg;base64,{image_data}"
                    }
                ]
            }
        ],
        max_output_tokens=500
    )
    
    # 3. 간소화된 변수로 출력 결과 반환
    return response.output_text

In [27]:
analyze_image("data/dogcat.jpg")

'이 이미지는 분홍색 배경 앞에 앉아 있는 네 마리의 동물들로 구성되어 있습니다. 왼쪽부터 시작해 보면, 첫 번째는 회색 고양이, 그 옆에는 귀엽고 포동포동한 회색 푸들이 있습니다. 중간에는 웃고 있는 코기 강아지가 있으며, 마지막으로 오른쪽에는 또 다른 고양이가 있습니다. 이들은 모두 귀엽고 분위기가 아늑한 모습으로 촬영되었습니다.'

**📝 정리** — 실제 이미지(`data/dogcat.jpg`)와 비교해 보니 분홍 배경, 왼쪽부터 고양이·회색 푸들·코기·고양이 순서까지 정확했다. 이미지는 base64로 인코딩해 `input_image`로 보낸다. (이미지 파일은 출처를 몰라 저장소에 넣지 않음)

**gpt-4o-mini는 멀티모달 텍스트 모델**

- 이미지 분석 (가능): 입력창으로 들어온 이미지 데이터를 디지털 픽셀 단위로 쪼개어 텍스트 문맥과 연결해 이해할 수 있습니다.

- 이미지 생성 (불가능): 이 모델의 최종 출력(Output)은 오직 텍스트(글자)와 코드뿐입니다. 화가처럼 픽셀을 조합해 그림 파일(.png, .jpg)을 만들어내는 인공지능 엔진이 내부에 탑재되어 있지 않기 때문입니다.

In [28]:
from openai import OpenAI

client = OpenAI()

def generate_support_image():
    print("이미지를 생성 중입니다...")
    try:
        response = client.images.generate(
            model="gpt-image-2",
            prompt="밝고 따스한 봄을 느낄 수 있는 수채화 같은 이미지",
            size="1024x1024",
            quality="high",   
            n=1
        )
        image_base64 = response.data[0].b64_json
        with open("output/spring_watercolor.png", "wb") as f:
            import base64
            f.write(base64.b64decode(image_base64))
        print("생성 완료")
    except Exception as e:
        print(e)

generate_support_image()

이미지를 생성 중입니다...
생성 완료


In [29]:
from openai import OpenAI

# 1. 클라이언트 초기화
client = OpenAI()

# 2. 신형 Responses API 호출 (stream=False가 기본값입니다)
response = client.responses.create(
    model="gpt-4o-mini",
    input="딥러닝에 대해 요약해줘.",  # 💡 messages 대신 단순 input 사용 가능!
    stream=False  # 한 번에 받기 위해 False로 설정 (생략 가능)
)

# 3. 결과 출력
# 복잡한 choices[0] 구조 없이 output_text로 바로 가져옵니다.
print("AI 응답: ", end="")
print(response.output_text)

AI 응답: 딥러닝(Deep Learning)은 인공신경망(Artificial Neural Networks)을 활용한 기계 학습의 한 분야로, 대규모 데이터에서 패턴을 학습하고 예측하는 데 사용됩니다. 주로 다음과 같은 특징을 가지고 있습니다:

1. **신경망 구조**: 입력층, 은닉층, 출력층으로 구성된 다층 신경망을 사용하여 데이터의 복잡한 패턴을 모델링합니다.

2. **비지도 학습 및 지도 학습**: 딥러닝은 레이블이 있는 데이터(지도 학습)와 레이블이 없는 데이터(비지도 학습) 모두에서 사용할 수 있습니다.

3. **특징 추출**: 전통적인 기계 학습 기법보다 특성(특징)을 직접 추출할 수 있는 능력이 뛰어나, 수작업으로 특징을 설계할 필요가 줄어듭니다.

4. **대량 데이터**: 대량의 데이터와 강력한 계산 능력(예: GPU)을 활용하여 성능을 극대화합니다.

5. **응용 분야**: 이미지 인식, 자연어 처리, 음성 인식, 자율주행차 등 다양한 분야에서 활용됩니다.

딥러닝은 최근 AI 발전의 중요한 기반 기술 중 하나로 자리 잡고 있습니다.


In [30]:
from openai import OpenAI

# 1. 클라이언트 초기화
client = OpenAI()

# 2. 신형 Responses API 호출 (stream=True 설정)
stream = client.responses.create(
    model="gpt-4o-mini",
    input="딥러닝에 대해 요약해줘.",
    stream=True,  # 스트리밍 활성화
)

print("AI 응답: ", end="")

# 3. 루프를 돌며 이벤트 기반 스트리밍 처리
for event in stream:
    # 💡 이벤트의 종류(type)가 텍스트 글자 조각인 경우에만 출력하도록 제한합니다.
    if event.type == "response.output_text.delta":
        # 최신 API에서는 event.delta에 곧바로 글자가 담겨 옵니다.
        print(event.delta, end="", flush=True)

AI 응답: 딥러닝(Deep Learning)은 인공신경망(Artificial Neural Networks)을 기반으로 한 기계 학습의 한 분야로, 데이터에서 자동으로 특징을 학습하여 문제를 해결하는 기술입니다. 주로 이미지 인식, 자연어 처리, 음성 인식 등의 분야에서 활용됩니다.

### 주요 특징:

1. **다층 구조**: 여러 개의 은닉층을 가진 신경망을 사용하여 복잡한 패턴을 학습.
  
2. **대량의 데이터 필요**: 학습을 위해 대량의 데이터가 필요하며, 데이터가 많을수록 모델의 성능이 향상됨.

3. **비선형성**: 비선형 활성화 함수를 통해 복잡한 함수 근사 가능.

4. **계산 능력**: GPU(그래픽 처리 장치)를 활용하여 대규모 연산을 수행.

### 주요 기술:

- **컨볼루션 신경망(CNN)**: 이미지 처리에 주로 사용.
- **순환 신경망(RNN)**: 시퀀스 데이터(예: 텍스트, 음성) 처리에 효과적.
- **변분 오토인코더(VAE)** 및 **생성적 적대 신경망(GAN)**: 새로운 데이터를 생성하는 데 사용.

### 적용 분야:

- 자율주행차
- 의료 진단
- 추천 시스템
- 챗봇 및 가상 비서

딥러닝은 데이터 기반의 문제 해결에 강력한 도구로 자리잡고 있으며, 지속적으로 발전하고 있습니다.

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

In [32]:
from openai import OpenAI

client = OpenAI()

def generate_support_image():
    print("이미지를 생성 중입니다...")
    try:
        response = client.images.generate(
            model="gpt-image-2",
            prompt="밝고 따스한 봄을 느낄 수 있는 수채화 같은 이미지",
            size="1024x1024",
            quality="high",   
            n=1
        )
        image_base64 = response.data[0].b64_json
        with open("output/spring_watercolor.png", "wb") as f:
            import base64
            f.write(base64.b64decode(image_base64))
        print("생성 완료")
    except Exception as e:
        print(e)

generate_support_image()

이미지를 생성 중입니다...
생성 완료


**📝 정리** — `gpt-image-2`로 만든 이미지를 `output/spring_watercolor.png`에 저장했다. 벚꽃길, 유채꽃, 강이 있는 수채화가 나왔다.

![생성된 이미지](output/spring_watercolor.png)

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

**임베딩(Embedding)**
- 임베딩(Embedding)은  **컴퓨터가 단어나 문장의 의미를 이해할 수 있도록, 숫자의 배열(좌표)로 바꾸는 기술**입니다.
- 의미가 비슷한 단어나 문장은 공간상에서 가까운 위치에 모이게 되고, 의미가 다르면 멀리 떨어지게 됩니다.
- text-embedding-3-small은 OpenAI가 제공하는 가성비와 성능을 모두 잡은 최신 표준 임베딩 모델입니다.(1,536차원)
- 임베딩은 텍스트를 의미가 담긴 수치적 좌표로 변환하는 작업
- 임베딩은 단어를 숫자로 변환하는 것에서 끝나는 것이 아니라, 단어 사이의 의미적 관계를 벡터의 방향과 거리로 표현
  --> 임베딩 공간의 의미적 구조(Semantic Structure)

In [33]:
from openai import OpenAI
import numpy as np

client = OpenAI()

def get_embedding(text):
    resp = client.embeddings.create(
        input=text, model='text-embedding-3-small')
    return np.array(resp.data[0].embedding)

# 의미 유사도 실험
words = ['강아지', '개', '애견', '고양이', '사과', '비행기']
embeddings = {w: get_embedding(w) for w in words}

for key in embeddings :
    print(key, end="")
    print("(", len(embeddings[key]), ")", sep="")
    print(embeddings[key])

강아지(1536)
[ 0.00736618 -0.00386429  0.0141449  ...  0.01192474 -0.00159264
  0.00126553]
개(1536)
[ 0.05368042 -0.01472473  0.00736237 ...  0.00455475 -0.00191402
 -0.02009583]
애견(1536)
[ 0.03125     0.04876709  0.01980591 ...  0.01803589 -0.01287079
  0.00738525]
고양이(1536)
[ 0.00010866 -0.01676941 -0.06317139 ...  0.02444458 -0.03720093
 -0.02389526]
사과(1536)
[-0.01750183 -0.00103283  0.04260254 ... -0.0307312  -0.00307083
 -0.02481079]
비행기(1536)
[-0.03512573  0.00889587 -0.00468445 ...  0.00728989 -0.03503418
 -0.00782013]


**코사인 유사도**

(A · B) / (||A|| × ||B||)

A · B : 두 벡터의 내적(dot product), ||A||, ||B|| : 각 벡터의 크기(L2 norm)

- 벡터의 길이는 무시하고 방향만 비교합니다. 그래서 문서가 길든 짧든, 같은 기준으로 비교할 수 있어요.
- 값의 범위는 -1(완전히 반대 방향) ~ 1(완전히 같은 방향)이고, 0이면 두 벡터가 서로 직각(무관)입니다.
- 텍스트 임베딩 모델 출력은 보통 양수 공간에 분포해서, 실제로는 0~1 사이 값으로 구성됩니다.

In [34]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

base = "강아지"
embase = get_embedding(base)

for w in words:
    sim = cosine_similarity(embase, embeddings[w])
    print(f"{base} vs {w}: {sim:.4f}")

강아지 vs 강아지: 1.0000
강아지 vs 개: 0.2904
강아지 vs 애견: 0.2935
강아지 vs 고양이: 0.3361
강아지 vs 사과: 0.2071
강아지 vs 비행기: 0.2407


**📝 정리** — '강아지'와의 코사인 유사도가 '개'(0.29)·'애견'(0.29)보다 '고양이'(0.34)가 더 높게 나왔다. 단어 하나만 임베딩하면 문맥이 없어서 이런 일이 생긴다. '개'는 '사과 5개'의 '개'로도 쓰이는 등 뜻이 여러 개라 더 그렇다. 임베딩 유사도는 문장 단위에서 더 잘 맞는다.

### 텍스트 임베딩 공간에서는 여러 가지 흥미로운 성질이 나타납니다.
- 유사도 : 코사인 유사도 등으로 두 벡터의 각도를 계산하면 의미적 유사도를 근사할 수 있습니다.
- 군집 구조 : 비슷한 의미·주제의 단어/문장들이 자연스럽게 클러스터를 형성합니다.   
- 벡터(왕)−벡터(남자)+벡터(여자)≈벡터(여왕) : 남자 → 여자라는 의미 변화량을 왕에 적용하는 것입니다.
- 이것이 가능한 이유는 임베딩 모델이 학습 과정에서 단어들의 사용 패턴을 관찰하기 때문이며 모델은 이런 통계적 패턴을 수많은 문장에서 학습하면서 단어 간 관계를 벡터 공간에 반영하게 됩니다.

In [35]:
import os
import numpy as np
from openai import OpenAI

client = OpenAI()

MODEL = "text-embedding-3-small"

words = [
    "king",
    "queen",
    "man",
    "woman",
    "prince",
    "princess",
    "boy",
    "girl",
    "male",
    "female",
    "father",
    "mother",
    "husband",
    "wife",
    "emperor",
    "empress",
    "duke",
    "duchess",
    "actor",
    "actress",
    "doctor",
    "nurse",
    "teacher",
    "student",
    "apple",
    "car",
    "city",
    "computer"
]

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

response = client.embeddings.create(
    model=MODEL,
    input=words
)

embeddings = {
    word: np.array(item.embedding, dtype=np.float32)
    for word, item in zip(words, response.data)
}

target = embeddings["king"] - embeddings["man"] + embeddings["woman"]

scores = []
for word, vec in embeddings.items():
    if word in ["king", "man", "woman"]:
        continue
    score = cosine_similarity(target, vec)
    scores.append((word, score))

scores.sort(key=lambda x: x[1], reverse=True)

print("Query: king - man + woman")
print()
print("Top matches:")
for word, score in scores[:10]:
    print(f"{word:12s}  {score:.4f}")

print()
print("Similarity to queen:")
print(cosine_similarity(target, embeddings["queen"]))

Query: king - man + woman

Top matches:
queen         0.6162
female        0.4883
princess      0.4646
mother        0.4549
duchess       0.4497
girl          0.4379
empress       0.4265
wife          0.4230
prince        0.3907
emperor       0.3845

Similarity to queen:
0.61623627


In [36]:
words = [
    "왕",
    "여왕",
    "남자",
    "여자",
    "왕자",
    "공주",
    "황제",
    "여제",
    "아버지",
    "어머니",
    "남편",
    "아들",
    "소년",
    "소녀",
    "사과",
    "자동차",
    "컴퓨터"
]

response = client.embeddings.create(
    model=MODEL,
    input=words
)

embeddings = {
    word: np.array(item.embedding, dtype=np.float32)
    for word, item in zip(words, response.data)
}

target = embeddings["왕"] - embeddings["남자"] + embeddings["여자"]

scores = []
for word, vec in embeddings.items():
    if word in ["왕", "남자", "여자"]:
        continue
    score = cosine_similarity(target, vec)
    scores.append((word, score))

scores.sort(key=lambda x: x[1], reverse=True)

print("Query: 왕 - 남자 + 여자")
print()
print("Top matches:")
for word, score in scores[:10]:
    print(f"{word:8s} {score:.4f}")

print()
print("Similarity to 여왕:")
print(cosine_similarity(target, embeddings["여왕"]))

Query: 왕 - 남자 + 여자

Top matches:
여왕       0.5816
왕자       0.4880
여제       0.3843
어머니      0.2107
자동차      0.2099
사과       0.2066
황제       0.1977
공주       0.1933
소녀       0.1913
아버지      0.1467

Similarity to 여왕:
0.58157945


**📝 정리** — 영어 `king - man + woman`은 `queen`(0.62)이 1위였다. 한국어 `왕 - 남자 + 여자`도 `여왕`(0.58)이 1위였지만, `자동차`·`사과`가 `황제`·`공주`보다 위에 올라왔다. 이 모델의 한국어 단어 임베딩은 영어보다 의미 구조가 덜 뚜렷했다.

**임베딩은 단어마다 의미를 나타내는 주소를 부여하는 기술입니다.**

### 프롬프트 테스트

#### 시스템 프롬프트와 사용자(유저) 프롬프트

In [37]:
from openai import OpenAI

# 환경 변수(OPENAI_API_KEY)를 자동으로 로드하므로 인자를 비워둡니다.
client = OpenAI()

# System Instruction (AI의 역할 및 규칙 설정)
instructions_prompt: str = """
당신은 Python 시니어 개발자입니다.
코드 리뷰 시 버그·성능·가독성 순으로 피드백하세요.
한국어로 답변하고, 마크다운 형식을 사용하세요.
""".strip()

# User Input (실제 사용자 요청)
user_input: str = """
다음 코드를 리뷰해주세요:
def get_users(db):
    users = []
    for id in range(1000):
        users.append(db.query(id))
    return users
""".strip()

# 최신 Responses API 호출
# (참고: 대화형이나 복잡한 롤 분리가 필요한 경우 input에 리스트 형태로 [ {"role": "user", "content": ...} ] 전달도 가능합니다)
response = client.responses.create(
    model="gpt-4o-mini",
    instructions=instructions_prompt,  # system_prompt 역할이 파라미터로 명시 분리됨
    input=user_input,                  # 단순 단발성 요청은 문자열 그대로 대입 가능
    temperature=0.3,
    max_output_tokens=1000
)

# Responses API의 가장 편리한 장점: .output_text로 결과에 바로 접근
print(response.output_text)


코드 리뷰를 진행하겠습니다.

### 1. 버그
- `db.query(id)`가 `id`에 대한 유효한 사용자 정보를 반환하는지 확인해야 합니다. 만약 `id`가 유효하지 않다면 예외가 발생할 수 있습니다. 이를 방지하기 위해 예외 처리를 추가하는 것이 좋습니다.

### 2. 성능
- 현재 코드는 1000개의 사용자 정보를 순차적으로 쿼리하고 있습니다. 이는 데이터베이스에 대한 많은 요청을 발생시키며 성능 저하를 초래할 수 있습니다. 
- 가능하다면, 한 번의 쿼리로 여러 사용자를 가져오는 방법을 고려해보세요. 예를 들어, `db.query_multiple(ids)`와 같은 메서드를 사용하여 한 번에 여러 사용자를 가져오는 것이 좋습니다.

### 3. 가독성
- `users` 리스트를 초기화한 후, `for` 루프 내에서 `append`를 사용하는 대신 리스트 컴프리헨션을 사용할 수 있습니다. 이는 코드의 가독성을 높여줍니다.

### 개선된 코드 예시
```python
def get_users(db):
    try:
        # ids 리스트를 생성
        ids = range(1000)
        # 리스트 컴프리헨션을 사용하여 사용자 정보를 가져옴
        users = [db.query(id) for id in ids]
        return users
    except Exception as e:
        # 예외 처리
        print(f"Error occurred: {e}")
        return []
```

### 추가 제안
- `db.query` 메서드가 반환하는 값이 None일 경우를 처리하는 로직을 추가하는 것도 고려해보세요.
- 로그를 남기는 방법도 좋습니다. 예외 발생 시 어떤 `id`에서 문제가 발생했는지 기록하면 디버깅에 도움이 됩니다.


In [38]:
# Few-shot: messages 배열에 예시 대화를 추가
messages = [
    {'role': 'system', 'content': '감성 분류기. 긍정/중립/부정으로만 답합니다.'},
    # 예시 1
    {'role': 'user',      'content': '배송이 정말 빨랐어요!'},
    {'role': 'assistant', 'content': '긍정'},
    # 예시 2
    {'role': 'user',      'content': '보통이에요, 딱히 좋지도 나쁘지도 않아요'},
    {'role': 'assistant', 'content': '중립'},
    # 예시 3
    {'role': 'user',      'content': '포장이 엉망이고 제품에 흠집이 있었어요'},
    {'role': 'assistant', 'content': '부정'},
    # 분류할 리뷰
    {'role': 'user', 'content': '색상은 사진과 달랐지만 품질은 좋았어요'},
]

response = client.responses.create(
    model="gpt-4o-mini",
    input=messages,                  
    temperature=0,
)

# Responses API의 가장 편리한 장점: .output_text로 결과에 바로 접근
print(response.output_text)

긍정


#### CoT
- Chain-of-Thought의 약자로 LLM에게 바로 답만 내게 하는 것이 아니라, 문제를 단계별로 나누어 생각하게 만드는 프롬프트 기법
- "단계별로 xxx 해줘"

In [39]:
import json
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY")
)

system = '''
당신은 데이터 분석가입니다.
반드시 단계적으로 생각한 후(CoT) JSON으로만 출력하세요.
마크다운, 설명문은 포함하지 마세요.
'''

user = '''
다음 매출 데이터를 분석하세요:
1월: 1200만원, 2월: 980만원, 3월: 1450만원, 4월: 1600만원, 5월:1550만원

단계: 1)평균 계산 -> 2)최대/최소 -> 3)추세 -> 4)인사이트

JSON 스키마:
{"average": number, "max_month": string,
 "min_month": string, "trend": "상승|하락|보합",
 "insight": string}
'''

resp = client.responses.create(
    model='gpt-4o-mini',
    input=[{'role':'system','content':system},
              {'role':'user','content':user}],
    temperature=0
)

result = json.loads(resp.output_text)
print(f'평균: {result["average"]:,}만원')
print(f'최대매출: {result["max_month"]}')
print(f'최저매출: {result["min_month"]}')
print(f'추세: {result["trend"]}')
print(f'인사이트: {result["insight"]}')


평균: 1,340만원
최대매출: 4월
최저매출: 2월
추세: 상승
인사이트: 매출이 2월을 제외하고 지속적으로 증가하고 있으며, 4월에 가장 높은 매출을 기록함.


**📝 정리** — CoT를 시키면서 JSON으로만 답하게 하고 `json.loads()`로 바로 파싱했다. 평균 1,340만원, 최대 4월, 최소 2월로 계산이 맞았다. 다만 모델이 JSON 앞뒤에 다른 글을 붙이면 `json.loads()`가 실패하므로, 실제로는 구조화 출력 기능을 쓰는 게 안전하다.

In [40]:
from openai import OpenAI

client = OpenAI()  # API 키는 환경변수에서 자동 로드

response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[
        {'role': 'system', 'content': '당신은 친절한 AI 어시스턴트입니다.'},
        {'role': 'user', 'content': '파이썬이란 무엇인가요?'}
    ],
    max_tokens=500,
    temperature=0.7
)

print(response.choices[0].message.content)


파이썬(Python)은 고수준의 프로그래밍 언어로, 1991년 귀도 반 로썸(Guido van Rossum)에 의해 처음 개발되었습니다. 파이썬은 다음과 같은 특징을 가지고 있습니다:

1. **간결하고 읽기 쉬운 문법**: 파이썬은 코드가 명확하고 직관적입니다. 이는 초보자부터 전문가까지 쉽게 이해하고 사용할 수 있게 해줍니다.

2. **다양한 용도**: 웹 개발, 데이터 분석, 인공지능, 머신러닝, 과학 계산, 자동화 스크립트 등 다양한 분야에서 사용됩니다.

3. **광범위한 라이브러리와 프레임워크**: 파이썬은 수많은 라이브러리와 프레임워크(예: Django, Flask, NumPy, Pandas, TensorFlow 등)를 제공하여 개발자들이 복잡한 작업을 쉽게 수행할 수 있도록 돕습니다.

4. **크로스 플랫폼**: 파이썬은 Windows, macOS, Linux 등 다양한 운영 체제에서 사용할 수 있습니다.

5. **커뮤니티와 지원**: 파이썬은 활발한 개발자 커뮤니티가 있으며, 다양한 온라인 자료와 문서, 포럼을 통해 지원을 받을 수 있습니다.

이러한 특성 덕분에 파이썬은 프로그래밍 입문자에게 인기가 높고, 많은 기업과 연구 기관에서도 널리 사용되고 있습니다.


In [41]:
from openai import OpenAI

# SDK가 환경 변수(OPENAI_API_KEY)를 자동으로 찾으므로 인자는 생략 가능합니다.
client = OpenAI()

# 최신 Responses API 호출 방식
response = client.responses.create(
    model="gpt-4o-mini",
    
    # 1. 기존의 'system' 역할을 명시적인 파라미터로 처리
    instructions="당신은 친절한 AI 어시스턴트입니다.",
    
    # 2. 유저 질문을 복잡한 딕셔너리 없이 문자열 형태로 바로 대입
    input="파이썬이란 무엇인가요?",
    
    # 기존 파라미터 유지 (max_tokens -> Responses API에서는 max_tokens로 하위 호환 적용)
    max_output_tokens=500,
    temperature=0.7
)

# 3. .output_text 속성을 사용하여 데이터에 바로 접근 (가장 편리해진 부분)
print(response.output_text)

파이썬(Python)은 고급 프로그래밍 언어로, 1991년 귀도 반 로섬(Guido van Rossum)에 의해 처음 개발되었습니다. 파이썬은 코드가 간결하고 읽기 쉬워, 초보자부터 전문가까지 폭넓게 사용됩니다. 주요 특징은 다음과 같습니다.

1. **간결한 문법**: 코드가 직관적이고 이해하기 쉬워, 빠르게 배울 수 있습니다.
2. **다양한 용도**: 웹 개발, 데이터 분석, 인공지능, 머신러닝, 자동화 스크립트 등 다양한 분야에서 사용됩니다.
3. **강력한 라이브러리**: NumPy, pandas, TensorFlow, Django 등 다양한 라이브러리를 통해 기능을 확장할 수 있습니다.
4. **플랫폼 독립성**: Windows, macOS, Linux 등 다양한 운영 체제에서 실행할 수 있습니다.

이러한 특징 덕분에 파이썬은 많은 개발자와 데이터 과학자들 사이에서 인기가 높습니다.


**📝 정리** — 같은 질문을 `chat.completions`(위)와 `responses`(아래)로 보낸 비교. (참고: 주석에는 `max_tokens`로 하위 호환된다고 적혀 있지만, 실제 코드는 Responses API의 `max_output_tokens`를 쓴다)

In [43]:
import inspect
from openai import OpenAI

client = OpenAI()

def code_review(code: str, language: str = 'python') -> str:
    """
    최신 OpenAI Responses API를 사용하여 코드 리뷰를 수행하는 함수
    """
    
    # 1. 시스템 지침 정의 (기존 system role 역할)
    instructions_prompt = "시니어 개발자로서 코드를 리뷰합니다."
    
    # 2. 유저 프롬프트 정의 
    # (f-string 내부에 들여쓰기가 그대로 들어가는 것을 방지하기 위해 inspect.cleandoc 사용)
    user_prompt = inspect.cleandoc(f"""
        다음 {language} 코드를 리뷰해주세요:
        1. 버그 가능성
        2. 성능 개선점
        3. 보안 취약점
        4. 코드 품질/가독성

        ```{language}
        {code}
        ```
    """)
    
    # 3. 최신 API 구조로 호출
    review = client.responses.create(
        model='gpt-4o-mini',
        instructions=instructions_prompt,  # 명시적 시스템 프롬프트 분리
        input=user_prompt,                # 단발성 텍스트 입력 처리
    )
    
    # 4. .output_text 속성을 통해 최종 자연어 리뷰 결과 반환
    return review.output_text


**📝 정리** — `code_review()` 함수는 정의만 하고 호출하지 않아서 출력이 없다.

In [44]:
def translate_batch(texts, target_lang='English', source_lang='Korean'):
    results = []
    for text in texts:
        response = client.responses.create(
            model='gpt-4o-mini',
            input= f'{source_lang}를 {target_lang}로 번역하세요. 번역문만 출력:\n{text}',           
            max_output_tokens=200
        )
        results.append(response.output_text)
    return results

# 사용 예시
texts = ['안녕하세요', '감사합니다', '도움이 필요합니다']
translated = translate_batch(texts)
print(translated)

['Hello', 'Thank you.', 'I need help.']


<hr style="background-color:red; height:3px">

### <span style='color:blue; font-weight:bold'>파이썬기반 LLM 서비스 개발 기초</span>에 대해 학습했습니다.